# Step 5. Profile diagnostics (thermocline / halocline / pycnocline / oxycline / DCM / MLD)

Computes vertical-structure diagnostics from the 1 m downcast CTD profiles (and merged nitrate where
present), then writes a per-cast table and **annotated profile plots** with shaded interpretation
bands and compact literature citations — ported from the v1 step06 notebook.

> **Interpretation disclaimer.** These diagnostics are produced automatically from single profiles.
> The shaded bands mark *interpreted* zones, not exact physical layers. Cline depths are the location
> of the strongest smoothed gradient and are method- and smoothing-dependent; MLD is threshold-based.
> Always check the profile shape and the exported table before using these values scientifically.

## Diagnostics and their literature basis

| Feature | Method | Citation (compact) |
|---------|--------|--------------------|
| Thermocline | depth of max \|dT/dz\| (symmetric band) | Fiedler 2010; Chu & Fan 2019 |
| Halocline | depth of max \|dS/dz\| | gradient diagnostic |
| Pycnocline | depth of max \|dσ/dz\| | Chu & Fan 2019 |
| Oxycline | depth of most negative dO₂/dz | Fiedler 2010; Zhou et al. 2023 |
| DCM (Fmax) | smoothed fluorescence maximum | Cullen 1982; Falkowski & Kiefer 1985 |
| MLD (temperature) | first depth where \|T − T(10 m)\| ≥ 0.2 °C (surface-to-depth band) | de Boyer Montégut et al. 2004; Kara et al. 2000 |
| MLD (density) | first depth where σ − σ(10 m) ≥ 0.03 kg/m³ | de Boyer Montégut et al. 2004 |

Cline/DCM features render as a symmetric band of half-width 2 m around the detected depth; MLD renders
as a shaded surface-to-depth zone. Full references are listed in the final section.


## 1. Imports and settings

In [1]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)


In [2]:
# ===========================================================================
# Paths (match step04 / v2 layout)
# ===========================================================================
CTD_ROOT     = Path(r"C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\ctd")
CRUISES_ROOT = CTD_ROOT / "cruises"

CRUISE_ID  = "P45_06"
CRUISE_DIR = CRUISES_ROOT / CRUISE_ID

L2_1M_DOWN   = CRUISE_DIR / "L2" / "CTD" / "CTD_1m_down"
L2_SUNA_ROOT = CRUISE_DIR / "L2" / "SUNA"
PLOT_ROOT    = CRUISE_DIR / "L2" / "CTD" / "profile_plots"
AUDIT_ROOT   = CRUISE_DIR / "_audit"
for d in (PLOT_ROOT, AUDIT_ROOT):
    d.mkdir(parents=True, exist_ok=True)

# ===========================================================================
# Diagnostic parameters (ported verbatim from v1 step06)
# ===========================================================================
SMOOTHING_WINDOW = 5                       # rolling-mean window (points)
MIN_POINTS_FOR_GRADIENT = 5                # need at least this many valid points
MIXED_LAYER_REFERENCE_DEPTH_M = 10.0       # reference depth for MLD
MIXED_LAYER_TEMPERATURE_THRESHOLD = 0.2    # deg C (de Boyer Montegut)
MIXED_LAYER_DENSITY_THRESHOLD = 0.03       # kg/m^3 (de Boyer Montegut)
SURFACE_EXCLUDE_M = 3.0                     # ignore depths above this when finding clines
                                           # (near-surface sensor spikes / soak artifacts)

# ---- Annotation appearance (ported from v1 step06) --------------------------
DIAGNOSTIC_BAND_ALPHA = 0.16               # shaded band transparency
DIAGNOSTIC_BAND_HALF_WIDTH_M = 2.0         # +/- band width around a cline/DCM depth
FIG_TITLE_SIZE = 11

# Feature spec library: label, render mode, colour (Okabe-Ito colourblind-safe),
# and the compact + full literature citation for each diagnostic.
DIAGNOSTIC_LIBRARY = {
    "thermocline_depth_m": {
        "label": "Thermocline", "mode": "symmetric_band", "color": "#D55E00",
        "citation": "Fiedler 2010; Chu & Fan 2019"},
    "mld_temp_m": {
        "label": "MLD T", "mode": "surface_to_depth_band", "color": "#666666",
        "citation": "de Boyer Montegut et al. 2004"},
    "halocline_depth_m": {
        "label": "Halocline", "mode": "symmetric_band", "color": "#0072B2",
        "citation": "gradient diagnostic"},
    "pycnocline_depth_m": {
        "label": "Pycnocline", "mode": "symmetric_band", "color": "#CC79A7",
        "citation": "Chu & Fan 2019"},
    "mld_density_m": {
        "label": "MLD rho", "mode": "surface_to_depth_band", "color": "#8C8C8C",
        "citation": "de Boyer Montegut et al. 2004"},
    "oxycline_depth_m": {
        "label": "Oxycline", "mode": "symmetric_band", "color": "#009E9E",
        "citation": "Fiedler 2010; Zhou et al. 2023"},
    "dcm_depth_m": {
        "label": "Fmax (DCM)", "mode": "symmetric_band", "color": "#009E73",
        "citation": "Cullen 1982; Falkowski & Kiefer 1985"},
    "nitracline_onset_m": {
        "label": "Nitracline (onset)", "mode": "symmetric_band", "color": "#E69F00",
        "citation": "surface-min +1 uM; cf. Omand & Mahadevan 2015"},
}

# Nitrate smoothing window (points) for the trend curve overlay.
NITRATE_SMOOTH_WINDOW = 5

# Nitracline onset: first depth where smoothed nitrate exceeds the near-surface
# minimum by this amount (uM). Relative (not absolute) so it is robust to the raw
# SUNA offset bias. This is the ECOLOGICALLY meaningful nitracline (top of the rise);
# the peak-gradient depth is also reported for consistency with the other clines.
NITRACLINE_ONSET_DELTA = 1.0        # uM above surface minimum
NITRACLINE_SURFACE_LAYER_M = 15.0   # depth range used to define the "surface minimum"

DISCLAIMER_NOTE = (
    "Shaded bands mark interpreted diagnostic zones, not exact layers.\n"
    "Cline depths are the strongest smoothed gradient (method-dependent);\n"
    "MLD is threshold-based. Verify against the profile and the exported table."
)

# Column candidates (v2 Sea-Bird names + merged nitrate).
DEPTH_CANDIDATES = ["depSM", "depSW", "prdM", "prDM", "prM", "pressure"]
TEMP_CANDIDATES  = ["tv290C", "t090C", "potemp090C"]
SAL_CANDIDATES   = ["sal00", "sal11"]
DENS_CANDIDATES  = ["sigma-\u00e900", "sigma-t00", "density00"]
OXY_CANDIDATES   = ["sbox0Mm/Kg", "sbeox0Mm/Kg", "sbeox0Mm/L"]
FLUO_CANDIDATES  = ["flECO-AFL", "flECO-AFL1"]

FIG_DPI = 200

print("Cruise      :", CRUISE_ID)
print("CTD 1m down :", L2_1M_DOWN, "(exists:", L2_1M_DOWN.exists(), ")")
print("Diagnostics ->", AUDIT_ROOT / "profile_diagnostics.csv")


Cruise      : P45_06
CTD 1m down : C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\ctd\cruises\P45_06\L2\CTD\CTD_1m_down (exists: True )
Diagnostics -> C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\ctd\cruises\P45_06\_audit\profile_diagnostics.csv


## 2. Readers and column pickers (shared with step04)

In [3]:
def parse_cnv(path: Path) -> pd.DataFrame:
    text = path.read_text(encoding="latin-1", errors="replace").splitlines()
    names: Dict[int, str] = {}
    data_start = None
    name_re = re.compile(r"#\s*name\s+(\d+)\s*=\s*([^:]+?)\s*[:=]", re.IGNORECASE)
    for idx, line in enumerate(text):
        s = line.strip()
        if s.startswith("#") or s.startswith("*"):
            m = name_re.search(line)
            if m:
                names[int(m.group(1))] = m.group(2).strip()
            if s.upper().startswith("*END*"):
                data_start = idx + 1
                break
    if data_start is None:
        raise ValueError(f"No *END* in {path}")
    cols = [names[k] for k in sorted(names)]
    rows = []
    for line in text[data_start:]:
        s = line.strip()
        if s and len(s.split()) >= len(cols):
            rows.append(s.split()[:len(cols)])
    return pd.DataFrame(rows, columns=cols).apply(pd.to_numeric, errors="coerce")


def pick(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    return None


## 3. Diagnostic functions (ported from v1)

Smoothing, vertical gradient, extreme-gradient depth, threshold-based MLD, and smoothed maximum — the
same logic v1 used, so the numbers are comparable to earlier work.

In [4]:
def _rolling_smooth(series: pd.Series, window: int = SMOOTHING_WINDOW) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").rolling(window=window, center=True, min_periods=1).mean()


def _gradient(values: pd.Series, depth: pd.Series) -> pd.Series:
    values = pd.to_numeric(values, errors="coerce")
    depth = pd.to_numeric(depth, errors="coerce")
    valid = pd.DataFrame({"v": values, "d": depth}).dropna()
    if len(valid) < MIN_POINTS_FOR_GRADIENT or valid["d"].nunique() < MIN_POINTS_FOR_GRADIENT:
        return pd.Series(np.nan, index=values.index)
    g = np.gradient(valid["v"].to_numpy(float), valid["d"].to_numpy(float))
    out = pd.Series(np.nan, index=values.index)
    out.loc[valid.index] = g
    return out


def _clean(df: pd.DataFrame, value_col: str, depth_col: str) -> pd.DataFrame:
    work = df[[value_col, depth_col]].apply(pd.to_numeric, errors="coerce").dropna()
    return work.sort_values(depth_col).reset_index(drop=True)


def extreme_gradient_depth(df, value_col, depth_col, mode="max_abs"):
    work = _clean(df, value_col, depth_col)
    # drop the near-surface layer (spikes/soak) before locating the cline
    work = work[work[depth_col] >= SURFACE_EXCLUDE_M].reset_index(drop=True)
    if len(work) < MIN_POINTS_FOR_GRADIENT:
        return np.nan, np.nan
    grad = _gradient(_rolling_smooth(work[value_col]), work[depth_col])
    if grad.isna().all():
        return np.nan, np.nan
    if mode == "max_positive":
        idx = grad.idxmax()
    elif mode == "max_negative":
        idx = grad.idxmin()
    else:
        idx = grad.abs().idxmax()
    return float(work.loc[idx, depth_col]), float(grad.loc[idx])


def smoothed_max_depth(df, value_col, depth_col):
    work = _clean(df, value_col, depth_col)
    if work.empty:
        return np.nan, np.nan
    sm = _rolling_smooth(work[value_col])
    if sm.isna().all():
        return np.nan, np.nan
    idx = sm.idxmax()
    return float(work.loc[idx, depth_col]), float(work.loc[idx, value_col])


def mld_threshold_depth(depth, metric, threshold, ref_depth, direction="absolute"):
    work = pd.DataFrame({"depth": pd.to_numeric(depth, errors="coerce"),
                         "metric": pd.to_numeric(metric, errors="coerce")}).dropna()
    if work.empty:
        return np.nan
    work = work.groupby("depth", as_index=False)["metric"].mean().sort_values("depth").reset_index(drop=True)
    deeper = work[work["depth"] >= ref_depth]
    ref_idx = deeper["depth"].idxmin() if not deeper.empty else (work["depth"] - ref_depth).abs().idxmin()
    ref_value = work.loc[ref_idx, "metric"]
    below = work[work["depth"] >= work.loc[ref_idx, "depth"]]
    if direction == "absolute":
        change = (below["metric"] - ref_value).abs()
    elif direction == "decrease":
        change = ref_value - below["metric"]
    else:
        change = below["metric"] - ref_value
    exceed = below[change >= threshold]
    return float(exceed.iloc[0]["depth"]) if not exceed.empty else np.nan


## 4. Compute diagnostics per cast

In [5]:
def compute_diagnostics(df: pd.DataFrame, cast_id: str) -> Dict[str, Any]:
    depth_col = pick(df, DEPTH_CANDIDATES)
    out: Dict[str, Any] = {"cast_id": cast_id, "depth_col": depth_col}
    if depth_col is None:
        return out
    depth = pd.to_numeric(df[depth_col], errors="coerce")

    tcol = pick(df, TEMP_CANDIDATES)
    if tcol:
        d, g = extreme_gradient_depth(df, tcol, depth_col, "max_abs")
        out["thermocline_depth_m"] = d
        out["dTdz_at_thermocline"] = g
        out["mld_temp_m"] = mld_threshold_depth(depth, df[tcol],
                             MIXED_LAYER_TEMPERATURE_THRESHOLD, MIXED_LAYER_REFERENCE_DEPTH_M, "absolute")

    scol = pick(df, SAL_CANDIDATES)
    if scol:
        d, g = extreme_gradient_depth(df, scol, depth_col, "max_abs")
        out["halocline_depth_m"] = d
        out["dSdz_at_halocline"] = g

    dcol = pick(df, DENS_CANDIDATES)
    if dcol:
        d, g = extreme_gradient_depth(df, dcol, depth_col, "max_abs")
        out["pycnocline_depth_m"] = d
        out["mld_density_m"] = mld_threshold_depth(depth, df[dcol],
                               MIXED_LAYER_DENSITY_THRESHOLD, MIXED_LAYER_REFERENCE_DEPTH_M, "increase")

    ocol = pick(df, OXY_CANDIDATES)
    if ocol:
        d, g = extreme_gradient_depth(df, ocol, depth_col, "max_negative")
        out["oxycline_depth_m"] = d

    fcol = pick(df, FLUO_CANDIDATES)
    if fcol:
        d, v = smoothed_max_depth(df, fcol, depth_col)
        out["dcm_depth_m"] = d
        out["dcm_fluor_value"] = v

    if "suna_nitrate" in df.columns and df["suna_nitrate"].notna().any():
        # (a) onset: first depth where smoothed nitrate exceeds the surface minimum
        #     by NITRACLINE_ONSET_DELTA -- the ecological "top of the nitracline".
        nd = df[["suna_nitrate", depth_col]].apply(pd.to_numeric, errors="coerce").dropna()
        nd = nd.sort_values(depth_col).reset_index(drop=True)
        if len(nd) >= MIN_POINTS_FOR_GRADIENT:
            sm = nd["suna_nitrate"].rolling(NITRATE_SMOOTH_WINDOW, center=True, min_periods=1).mean()
            surf = sm[nd[depth_col] <= NITRACLINE_SURFACE_LAYER_M]
            surf_min = float(surf.min()) if len(surf) else float(sm.min())
            crossed = nd[sm >= surf_min + NITRACLINE_ONSET_DELTA]
            out["nitracline_onset_m"] = float(crossed[depth_col].iloc[0]) if not crossed.empty else np.nan
            out["nitracline_surface_min_uM"] = surf_min
        # (b) peak gradient: steepest nitrate increase (consistent with other clines)
        d, g = extreme_gradient_depth(df, "suna_nitrate", depth_col, "max_positive")
        out["nitracline_maxgrad_m"] = d
        out["dNdz_at_nitracline"] = g

    return out


# Load casts (same discovery as step04) and compute.
frames: Dict[str, pd.DataFrame] = {}
records: List[Dict[str, Any]] = []
for f in sorted(L2_1M_DOWN.glob("*_1m_down.cnv")):
    cast_id = f.name[: -len("_1m_down.cnv")]
    try:
        df = parse_cnv(f)
        # attach nitrate if merged product exists (depth-binned)
        merged = L2_SUNA_ROOT / f"{cast_id}_SUNA_1s.csv"
        if merged.exists():
            depth_col = pick(df, DEPTH_CANDIDATES)
            m = pd.read_csv(merged)
            md_ = pick(m, DEPTH_CANDIDATES)
            if md_ and "suna_nitrate" in m.columns and depth_col:
                mm = m[[md_, "suna_nitrate"]].apply(pd.to_numeric, errors="coerce").dropna()
                mm["_b"] = mm[md_].round().astype(int)
                binned = mm.groupby("_b")["suna_nitrate"].mean()
                df["suna_nitrate"] = df[depth_col].round().astype("Int64").map(binned)
        frames[cast_id] = df
        records.append(compute_diagnostics(df, cast_id))
    except Exception as exc:
        print(f"  {cast_id}: FAILED -> {exc!r}")

diag = pd.DataFrame(records)
diag_path = AUDIT_ROOT / "profile_diagnostics.csv"
diag.to_csv(diag_path, index=False, encoding="utf-8-sig")
display(diag)
print(f"\nDiagnostics for {len(diag)} casts -> {diag_path}")


,cast_id,depth_col,thermocline_depth_m,dTdz_at_thermocline,mld_temp_m,halocline_depth_m,dSdz_at_halocline,pycnocline_depth_m,mld_density_m,oxycline_depth_m,dcm_depth_m,dcm_fluor_value,nitracline_onset_m,nitracline_surface_min_uM,nitracline_maxgrad_m,dNdz_at_nitracline
0,P45_06_CTD_01,depSM,33.0,-0.44995,20.0,27.0,0.04296,27.0,18.0,30.0,41.0,0.8261,NaN,NaN,NaN,NaN
1,P45_06_CTD_02,depSM,33.0,-0.46077,24.0,28.0,0.06454,33.0,23.0,44.0,48.0,0.6984,NaN,NaN,NaN,NaN
2,P45_06_CTD_03,depSM,25.0,-0.51347,14.0,25.0,0.03540,25.0,12.0,34.0,32.0,0.8292,32.0,-1.534345,36.0,0.343012
3,P45_06_CTD_04,depSM,32.0,-0.42404,16.0,23.0,0.04146,23.0,13.0,40.0,39.0,0.9082,36.0,-0.720805,49.0,1.325212
4,P45_06_CTD_05,depSM,22.0,-0.45671,15.0,22.0,0.04737,22.0,12.0,33.0,39.0,0.8979,24.0,-1.694234,33.0,1.419907
5,P45_06_CTD_06,depSM,30.0,-0.48634,17.0,23.0,0.04868,30.0,16.0,40.0,45.0,0.8115,45.0,-0.548851,59.0,0.740346
6,P45_06_CTD_07,depSM,24.0,-0.51388,15.0,22.0,0.04412,24.0,13.0,31.0,35.0,1.0332,20.0,-1.502085,23.0,0.812729
7,P45_06_CTD_08,depSM,28.0,-0.56450,17.0,25.0,0.04735,28.0,16.0,35.0,38.0,1.2609,36.0,-0.998613,51.0,0.666056
8,P45_06_CTD_09,depSM,22.0,-0.39774,13.0,22.0,0.03047,22.0,12.0,24.0,38.0,1.2037,NaN,NaN,NaN,NaN
9,P45_06_CTD_10,depSM,32.0,-0.42588,17.0,26.0,0.03894,32.0,14.0,46.0,50.0,0.9380,NaN,NaN,NaN,NaN



Diagnostics for 11 casts -> C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\ctd\cruises\P45_06\_audit\profile_diagnostics.csv


## 5. Annotated profile plots (feature depths marked)

In [6]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D


def _draw_feature(ax, key, depth_value, depth_max):
    """Shade one diagnostic band on an axis. Returns (label, color, mode) for the legend."""
    spec = DIAGNOSTIC_LIBRARY.get(key)
    if spec is None or depth_value is None or not np.isfinite(depth_value):
        return None
    color = spec["color"]
    if spec["mode"] == "surface_to_depth_band":
        ax.axhspan(0.0, depth_value, color=color, alpha=0.10, lw=0)
        ax.axhline(depth_value, color=color, ls=(0, (1, 1)), lw=1.1, alpha=0.9)
    else:
        top = max(0.0, depth_value - DIAGNOSTIC_BAND_HALF_WIDTH_M)
        bot = min(depth_max, depth_value + DIAGNOSTIC_BAND_HALF_WIDTH_M)
        ax.axhspan(top, bot, color=color, alpha=DIAGNOSTIC_BAND_ALPHA, lw=0)
        ax.axhline(depth_value, color=color, ls="-.", lw=1.2, alpha=0.9)
    return (f"{spec['label']} \u2248 {depth_value:.1f} m ({spec['citation']})", color, spec["mode"])


def write_diagnostics_txt(cast_id, diag_row):
    """Write a clean, readable per-cast diagnostics sidecar."""
    lines = [f"Profile diagnostics \u2014 {cast_id}", "=" * (22 + len(cast_id)), ""]
    order = [
        ("thermocline_depth_m", "Thermocline", "m", "Fiedler 2010; Chu & Fan 2019"),
        ("halocline_depth_m",   "Halocline",   "m", "gradient diagnostic"),
        ("pycnocline_depth_m",  "Pycnocline",  "m", "Chu & Fan 2019"),
        ("oxycline_depth_m",    "Oxycline",    "m", "Fiedler 2010; Zhou et al. 2023"),
        ("dcm_depth_m",         "DCM (Fmax)",  "m", "Cullen 1982; Falkowski & Kiefer 1985"),
        ("nitracline_onset_m",  "Nitracline (onset)",   "m", "surface-min +1 uM; cf. Omand & Mahadevan 2015"),
        ("nitracline_maxgrad_m","Nitracline (max grad)","m", "steepest dN/dz; consistent w/ other clines"),
        ("mld_temp_m",          "MLD (temperature, 0.2 C)", "m", "de Boyer Montegut et al. 2004"),
        ("mld_density_m",       "MLD (density, 0.03 kg/m3)","m", "de Boyer Montegut et al. 2004"),
    ]
    for key, label, unit, cite in order:
        val = diag_row.get(key)
        if val is not None and np.isfinite(val):
            lines.append(f"  {label:<28} {val:6.1f} {unit}    [{cite}]")
        else:
            lines.append(f"  {label:<28}    n/a")
    lines += ["",
              "NOTE: Automated single-profile diagnostics. Cline depths are the location of the",
              "strongest smoothed gradient (method- and smoothing-dependent); MLD is threshold-based.",
              "Treat as first-pass guidance, not definitive layer boundaries. Verify against the",
              "profile shape before scientific use."]
    (PLOT_ROOT / f"{cast_id}_diagnostics.txt").write_text("\n".join(lines), encoding="utf-8")


def annotated_plot(df, cast_id, diag_row):
    depth_col = pick(df, DEPTH_CANDIDATES)
    if depth_col is None:
        return None
    depth_max = pd.to_numeric(df[depth_col], errors="coerce").max()

    panels = [
        (pick(df, TEMP_CANDIDATES), "Temperature", "\u00b0C", ["thermocline_depth_m", "mld_temp_m"], False),
        (pick(df, SAL_CANDIDATES),  "Salinity", "PSU", ["halocline_depth_m"], False),
        (pick(df, DENS_CANDIDATES), "Density", "kg/m\u00b3", ["pycnocline_depth_m", "mld_density_m"], False),
        (pick(df, OXY_CANDIDATES),  "Oxygen", "\u00b5mol/kg", ["oxycline_depth_m"], False),
        (pick(df, FLUO_CANDIDATES), "Fluorescence", "mg/m\u00b3", ["dcm_depth_m"], False),
    ]
    has_no3 = "suna_nitrate" in df.columns and df["suna_nitrate"].notna().any()
    if has_no3:
        panels.append(("suna_nitrate", "Nitrate", "\u00b5M", ["nitracline_onset_m"], True))
    panels = [p for p in panels if p[0] is not None]
    n = len(panels)
    if n == 0:
        return None

    # Grid: aim for 3 columns. 6 panels -> 2 rows x 3 cols; 5 -> 3 top, 2 bottom.
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axgrid = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 4.9 * nrows),
                               sharey=True, squeeze=False)
    axes_flat = axgrid.flatten()

    legend_seen = {}
    for ax, (xcol, label, unit, keys, is_nitrate) in zip(axes_flat, panels):
        d = df[[xcol, depth_col]].apply(pd.to_numeric, errors="coerce").dropna().sort_values(depth_col)
        if not d.empty:
            if is_nitrate:
                # raw points + smoothed trend curve
                ax.plot(d[xcol], d[depth_col], ".", ms=3, color="#4a90d9", alpha=0.55, label="raw")
                sm = d[xcol].rolling(NITRATE_SMOOTH_WINDOW, center=True, min_periods=1).mean()
                ax.plot(sm, d[depth_col], "-", lw=1.6, color="#0b3d66", label="smoothed")
            else:
                ax.plot(d[xcol], d[depth_col], "-", lw=1.3, color="#1f77b4")
        if is_nitrate:
            mg = diag_row.get("nitracline_maxgrad_m")
            if mg is not None and np.isfinite(mg):
                ax.axhline(mg, color="#b8860b", ls=":", lw=1.0, alpha=0.8)
        for key in keys:
            res = _draw_feature(ax, key, diag_row.get(key), depth_max)
            if res:
                lbl, color, mode = res
                legend_seen[lbl] = (color, mode)
        ax.set_xlabel(f"{label} ({unit})", fontsize=9)
        ax.xaxis.set_label_position("top"); ax.xaxis.tick_top()
        ax.grid(False)                                  # no gridlines
        ax.spines["right"].set_visible(False)
    # y label on left column only
    for r in range(nrows):
        axgrid[r, 0].set_ylabel("Depth (m)")
    axes_flat[0].invert_yaxis()

    # hide any unused axes
    for ax in axes_flat[n:]:
        ax.set_visible(False)

    # Legend box in the first unused cell if there is one, else below the figure.
    handles = [Patch(facecolor=c, edgecolor=c, alpha=min(0.9, DIAGNOSTIC_BAND_ALPHA * 2.2), label=l)
               for l, (c, m) in legend_seen.items()]
    if has_no3:
        handles = [Line2D([0], [0], color="#0b3d66", lw=1.6, label="Nitrate (smoothed)"),
                   Line2D([0], [0], marker=".", ls="", color="#4a90d9", label="Nitrate (raw)")] + handles

    if n < nrows * ncols:  # there is a free cell -> put legend there
        legend_ax = axes_flat[n]
        legend_ax.set_visible(True); legend_ax.axis("off")
        legend_ax.legend(handles=handles, loc="upper left", fontsize=8, frameon=True)
        legend_ax.text(0.0, 0.02, DISCLAIMER_NOTE, transform=legend_ax.transAxes,
                       fontsize=7, va="bottom", ha="left", color="#555555", style="italic")
    else:  # full grid -> legend under the figure
        fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=8,
                   frameon=True, bbox_to_anchor=(0.5, -0.06))
        fig.text(0.01, -0.12, DISCLAIMER_NOTE, fontsize=7, va="top", ha="left",
                 color="#555555", style="italic")

    fig.suptitle(cast_id, fontsize=FIG_TITLE_SIZE, y=1.01)
    out = PLOT_ROOT / f"{cast_id}__annotated.png"
    fig.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    write_diagnostics_txt(cast_id, diag_row)
    return out


diag_by_cast = {r["cast_id"]: r for r in records}
made = []
for cast_id, df in frames.items():
    p = annotated_plot(df, cast_id, diag_by_cast.get(cast_id, {}))
    if p:
        made.append(p)
print(f"Wrote {len(made)} annotated figures + diagnostics .txt sidecars to {PLOT_ROOT}")


Wrote 11 annotated figures + diagnostics .txt sidecars to C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\ctd\cruises\P45_06\L2\CTD\profile_plots


## 6. Literature basis

The diagnostic methods and MLD thresholds follow standard references. Compact tags in the plot legends
map to these:

- **de Boyer Montégut, C., Madec, G., Fischer, A. S., Lazar, A., & Iudicone, D. (2004).** Mixed layer
  depth over the global ocean: an examination of profile data and a profile-based climatology.
  *J. Geophys. Res. Oceans, 109,* C12003. doi:10.1029/2004JC002378.
- **Kara, A. B., Rochford, P. A., & Hurlburt, H. E. (2000).** An optimal definition for ocean mixed
  layer depth. *J. Geophys. Res. Oceans, 105*(C7), 16803–16821. doi:10.1029/2000JC900072.
- **Cullen, J. J. (1982).** The deep chlorophyll maximum: comparing vertical profiles of chlorophyll a.
  *Can. J. Fish. Aquat. Sci., 39*(5), 791–803. doi:10.1139/f82-108.
- **Falkowski, P. G., & Kiefer, D. A. (1985).** Chlorophyll a fluorescence in phytoplankton:
  relationship to photosynthesis and biomass. *J. Plankton Res., 7*(5), 715–731.
  doi:10.1093/plankt/7.5.715.
- **Fiedler, P. C. (2010).** Comparison of objective descriptions of the thermocline.
  *Limnol. Oceanogr.: Methods, 8,* 313–325. doi:10.4319/lom.2010.8.313.
- **Chu, P. C., & Fan, C. (2019).** Global ocean synoptic thermocline gradient, isothermal-layer depth,
  and other upper-ocean parameters. *Sci. Data, 6,* 119. doi:10.1038/s41597-019-0125-3.
- **Zhou, Y., et al. (2023).** Spatiotemporal variations of the oxycline and its response to physical
  forcing. (oxycline gradient diagnostic.)

**Reminder:** these are automated, single-profile diagnostics. Treat them as first-pass guidance, not
definitive layer boundaries.
